# Проект: Обучение с учителем: качество модели

Интернет-магазин «В один клик» столкнулся со снижением активности постоянных клиентов. Для решения этой проблемы необходимо разработать систему персонализированных предложений.

Цель проекта: построить модель, прогнозирующую вероятность снижения покупательской активности клиента, и на её основе выделить целевые сегменты для персонализированных маркетинговых стратегий.

Задачи:

1. Проанализировать данные о поведении клиентов;

2. Построить и выбрать лучшую модель классификации;

3. Определить ключевые факторы снижения активности;

4. Выделить сегменты клиентов и предложить меры по удержанию.

## 1. Загрузка данных

In [1]:
!pip install shap -q

<class 'OSError'>: Not available

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
try:
    import phik
    from phik.report import plot_correlation_matrix
    from phik import report
except ImportError:
    !pip install phik
    import phik
    from phik.report import plot_correlation_matrix
    from phik import report

In [ ]:
market_file = pd.read_csv('/datasets/market_file.csv')
market_money = pd.read_csv('/datasets/market_money.csv')
market_time = pd.read_csv('/datasets/market_time.csv')
money = pd.read_csv('/datasets/money.csv', sep = ';')

In [ ]:
market_file.head()

In [ ]:
market_file.info()

In [ ]:
market_money.head()

In [ ]:
market_money.info()

In [ ]:
market_time.head()

In [ ]:
market_time.info()

In [ ]:
money.head()

In [ ]:
money.info()

Даннные датафреймов соответствуют описаниюю задачи. 

market_file.csv - 1300 клиентов, 13 признаков (целевой-Покупательская активность)

market_money.csv - 3900 записей о выручке по периодам

market_time.csv - 2600 записей о времени на сайте по периодам

money.csv - 1300 записей о прибыльности клиентов


На первый взгляд можно сказать, что необходимо выполнить предобработку данных, так как есть опечатки , некорректные разделители.
В файле money.csv столбец Прибыль имеет тип object вместо числового (вероятно из-за разделителя десятичных дробей).
Данные о выручке и времени представлены в "длинном" формате (несколько строк на клиента), нужно преобразовать в "широкий"

Отсутствуют явные пропуски во всех таблицах

## Предобработка данных

### Предобработка данных в датафрейме market_file

In [ ]:
market_file.isna().sum() #проверка пропусков

In [ ]:
market_file.duplicated().sum() #проверка дубликатов

In [ ]:
#приведение названий столбцов к формату snake_case
market_file = market_file.rename(columns={
    'Покупательская активность':'Покупательская_активность',
    'Тип сервиса':'Тип_сервиса',
    'Разрешить сообщать':'Разрешить_сообщать'
})

market_file.columns.tolist()

In [ ]:
market_file['Тип_сервиса'].unique()  # проверка опечаток в данных

In [ ]:
market_file['Тип_сервиса'] = market_file['Тип_сервиса'].replace({'стандартт':'стандарт'})

### Предобработка данных в датафрейме market_money

In [ ]:
market_money.isna().sum() #проверка пропусков

In [ ]:
market_money.duplicated().sum() #проверка дубликатов

In [ ]:
market_money['Период'].unique() #проверка опечаток в данных

### Предобработка данных в датафрейме market_time

In [ ]:
market_time.isna().sum() #проверка пропусков

In [ ]:
market_time.duplicated().sum() #проверка дубликатов

In [ ]:
market_time['Период'].unique() 

In [ ]:
market_time['Период'] = market_time['Период'].replace({'предыдцщий_месяц':'предыдущий_месяц'})

### Предобработка данных в датафрейме money

In [ ]:
money.isna().sum() #проверка пропусков

In [ ]:
money.duplicated().sum() #проверка дубликатов

In [ ]:
col_to_convert_money = ['Прибыль']

for col in col_to_convert_money:
    if col in money.columns:
        money[col] = money[col].astype(str).str.replace(',','.').astype(float)
money.head()

## Исследовательский анализ данных

### Проведем анализ целевой переменной- Покупательская активность.  market_file

In [ ]:
# Считаем распределение классов
counts = market_file['Покупательская_активность'].value_counts()
percentages = market_file['Покупательская_активность'].value_counts(normalize=True) * 100

In [ ]:
print("Распределение клиентов:")
for activity, count in counts.items():
    percent = percentages[activity]
    print(f"- {activity}: {count} клиентов ({percent:.1f}%)")

In [ ]:
plt.figure(figsize=(4, 4))
plt.pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Уровень покупательской активности')
plt.show()

Классы несбалансированы. Для оценки модели лучше использовать F1-score.

### Проведем анализ категориальных признаков и посмотрим их распределение market_file.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 4)) #cоздаем subplot для трех графиков

#1. Тип сервиса
ax1 = axes[0]
service_counts = market_file['Тип_сервиса'].value_counts()
sns.barplot(x=service_counts.index, y=service_counts.values, ax=ax1, palette='viridis')
ax1.set_title('Распределение по типу сервиса', fontsize=14)
ax1.set_xlabel('Тип сервиса')
ax1.set_ylabel('Количество клиентов')


#2. Разрешить сообщать
ax2 = axes[1]
permission_counts = market_file['Разрешить_сообщать'].value_counts()
sns.barplot(x=permission_counts.index, y=permission_counts.values, ax=ax2, palette='viridis')
ax2.set_title('Распределение по согласию на рассылку', fontsize=14)
ax2.set_xlabel('Разрешить сообщать')
ax2.set_ylabel('Количество клиентов')


In [ ]:
#3. Популярная категория
figs = plt.subplots(1, 1, figsize=(18, 4))
category_counts = market_file['Популярная_категория'].value_counts()
sns.barplot(x=category_counts.index, y=category_counts.values, palette='viridis')

plt.tight_layout()
plt.show()

In [ ]:
#вывод статистики
print("Статистика по категориальным признакам")
print(f"1. Тип сервиса:")
for category, count in service_counts.items():
    percentage = (count / len(market_file)) * 100
    print(f"   - {category}: {count} клиентов ({percentage:.1f}%)")

print(f"\n2. Разрешить сообщать:")
for category, count in permission_counts.items():
    percentage = (count / len(market_file)) * 100
    print(f"   - {category}: {count} клиентов ({percentage:.1f}%)")

print(f"\n3. Популярная категория (топ-5):")
for i, (category, count) in enumerate(category_counts.head().items()):
    percentage = (count / len(market_file)) * 100
    print(f"   {i+1}. {category}: {count} клиентов ({percentage:.1f}%)")

print(f"\nВсего уникальных категорий товаров: {len(category_counts)}")

Клиентов со стандартным типом сервиса значительно больше, большинство клиентов дали согласие на рассылку (74%). Наиболее популярные категории: "Товары для детей", "Домашний текстиль", "Косметика и аксесуары". Всего 5 уникальных категорий товаров.

### Проведем анализ числовых признаков и посмотрим их распределение в market_file

In [ ]:
numerical_columns = market_file.select_dtypes(include=['int64', 'float64']).columns.tolist() #выделим числовые признаки (исключаем id и категориальные)
numerical_columns = [col for col in numerical_columns if col != 'id'] #убираем id, так как это идентификатор, а не признак

In [ ]:
numerical_columns

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 12))
axes = axes.ravel()  #преобразуем в плоский массив

for i, feature in enumerate(numerical_columns):
    ax = axes[i]
    
    sns.histplot(data=market_file, x=feature, kde=True, ax=ax, bins=30) #гистограмма с KDE
    
    # Среднее значение
    mean_val = market_file[feature].mean()
    ax.axvline(mean_val, color='red', linestyle='--', label=f'Среднее: {mean_val:.2f}')
    
    ax.set_title(f'{feature}', fontsize=12)
    ax.set_xlabel('')
    ax.legend()

plt.tight_layout()
plt.show()

У параметров Маркет_актив_6_мес и Маркет_актив_тек_мес распределение близко к нормальному, однако текущая маркетинговая активность немного ниже средней за 6 месяцев.

Распределение Длительности равномерное близко к нормальному, явные выбросы отсутствуют.

У параметра Акционные_покупки бимодальное распределение - пики у 0.2 и 1. Клиенты либо почти не покупают по акциям, либо покупают почти всё по акциям. Средняя доля акционных покупок - 32%.

Средний_просмотр_категорий_за_визит имеет нормальное распределение. Клиенты в среднем просматривают 4-5 категорий за визит.

Неоплаченные_продукты_штук_квартал распределение с длинным правым хвостом.

У параметра Ошибка_сервиса распределение близко к нормальному.

У параметра Страниц_за_визит скошенное распределение вправо.


### Отбор клиентов с покупательской активностью не менее трёх месяцев, то есть таких, которые что-либо покупали в этот период.

In [ ]:
market_money_wide = market_money.pivot_table(
    index='id',
    columns='Период',
    values='Выручка'
)

# Переименуем колонки для удобства
market_money_wide.columns = [f'выручка_{col}' for col in market_money_wide.columns]

all_3_months = ['выручка_текущий_месяц', 'выручка_предыдущий_месяц', 'выручка_препредыдущий_месяц']

#Клиенты, которые что-либо покупали хотя бы в одном из трех месяцев
active_condition = (
    (market_money_wide[all_3_months[0]] > 0) |
    (market_money_wide[all_3_months[1]] > 0) |
    (market_money_wide[all_3_months[2]] > 0)
)

#Разделяем клиентов
active_clients = market_money_wide[active_condition].index.tolist()
inactive_clients = market_money_wide[~active_condition].index.tolist()

print(f"Активных клиентов (покупали хотя бы раз за последние 3 месяца): {len(active_clients)}")
print(f"Неактивных клиентов (не покупали ни разу за последние 3 месяца): {len(inactive_clients)}")
print(f"Всего клиентов в исходных данных: {len(market_money_wide)}")


In [ ]:
#Проверяем, сколько клиентов покупали во всех трех месяцах
active_in_all_3 = (
    (market_money_wide[all_3_months[0]] > 0) &
    (market_money_wide[all_3_months[1]] > 0) &
    (market_money_wide[all_3_months[2]] > 0)
)
print(f"\nКлиентов, покупавших во всех 3 месяцах: {active_in_all_3.sum()}")

In [ ]:
active_clients = market_money_wide[active_in_all_3].index.tolist()
inactive_clients = market_money_wide[~active_in_all_3].index.tolist()

print(f"Активных клиентов (покупали во всех 3 месяцах): {len(active_clients)}")
print(f"Неактивных клиентов (не покупали хотя бы в одном месяце): {len(inactive_clients)}")
print(f"Всего клиентов в исходных данных: {len(market_money_wide)}")

In [ ]:
#Проверяем, сколько клиентов покупали в 2 из 3 месяцев
active_in_2_of_3 = (
    ((market_money_wide[all_3_months[0]] > 0) & (market_money_wide[all_3_months[1]] > 0)) |
    ((market_money_wide[all_3_months[0]] > 0) & (market_money_wide[all_3_months[2]] > 0)) |
    ((market_money_wide[all_3_months[1]] > 0) & (market_money_wide[all_3_months[2]] > 0))
)
print(f"Клиентов, покупавших в 2 из 3 месяцев: {active_in_2_of_3.sum()}")

In [ ]:
# Создаем датафрейм с активными клиентами для дальнейшей работы
active_in_2_of_3_clients = market_money_wide[active_in_2_of_3 & ~active_in_all_3].index.tolist()
print(f"\nКлиентов, покупавших только в 2 из 3 месяцев (исключая тех, кто купил во всех 3): {len(active_in_2_of_3_clients)}")

# Создаем датафрейм с активными клиентами (теми, кто покупал во всех 3 месяцах)
money_active = market_money_wide.loc[active_clients].reset_index()

# Проверяем среднюю выручку активных клиентов
print(f"\nСредняя выручка клиентов, активных во всех 3 месяцах:")
for period in all_3_months:
    avg_revenue = money_active[period].mean()
    median_revenue = money_active[period].median()
    total_revenue = money_active[period].sum()
    print(f"  {period}:")
    print(f"    Средняя: {avg_revenue:.2f}")
    print(f"    Медиана: {median_revenue:.2f}")
    print(f"    Общая выручка: {total_revenue:.2f}")

Фильтрация активных клиентов:

Из исходных 1300 клиентов активными (совершившими хотя бы одну покупку за последние 3 месяца) являются 1297 клиентов.

Неактивных клиентов 3

### Анализ таблицы money.

In [ ]:
# Статистика по прибыли
if money['Прибыль'].dtype != 'float64':
    print("\nИсправляем тип данных для столбца 'Прибыль'...")
    # Если прибыль хранится как строка, заменяем запятые на точки
    money['Прибыль'] = money['Прибыль'].astype(str).str.replace(',', '.').astype(float)

print("\nСтатистика по прибыльности клиентов:")
print(f"Средняя прибыль на клиента: {money['Прибыль'].mean():.2f}")
print(f"Медианная прибыль: {money['Прибыль'].median():.2f}")
print(f"Максимальная прибыль: {money['Прибыль'].max():.2f}")
print(f"Минимальная прибыль: {money['Прибыль'].min():.2f}")
print(f"Общая прибыль: {money['Прибыль'].sum():.2f}")

#Гистограмма распределения прибыли
plt.figure(figsize=(10, 6))
sns.histplot(data=money, x='Прибыль', kde=True, bins=30)
plt.title('Распределение прибыльности клиентов')
plt.xlabel('Прибыль')
plt.ylabel('Количество клиентов')
plt.axvline(money['Прибыль'].mean(), color='red', linestyle='--', label=f'Среднее: {money["Прибыль"].mean():.2f}')
plt.legend()
plt.show()

### Анализ таблицы market_money

In [ ]:
market_money['Выручка'].max()

In [ ]:
market_money['Выручка'].min()

In [ ]:
# удалим выбросы 
market_money = market_money[market_money['Выручка'] != 106862.2]

market_money = market_money[market_money['Выручка'] != 0]

# Статистика по выручке
print("Статистика по выручке")
print(f"Средняя выручка на запись: {market_money['Выручка'].mean():.2f}")
print(f"Медианная выручка: {market_money['Выручка'].median():.2f}")
print(f"Максимальная выручка: {market_money['Выручка'].max():.2f}")
print(f"Минимальная выручка: {market_money['Выручка'].min():.2f}")
print(f"Общая выручка: {market_money['Выручка'].sum():.2f}")
print(f"Стандартное отклонение: {market_money['Выручка'].std():.2f}")

# Гистограмма распределения выручки
plt.figure(figsize=(30,6))
plt.subplot(1, 2, 1)
sns.histplot(data=market_money, x='Выручка', kde=True, bins=50)
plt.title('Распределение выручки')
plt.xlabel('Выручка')
plt.ylabel('Количество записей')
plt.axvline(market_money['Выручка'].mean(), color='red', linestyle='--', 
            label=f'Среднее: {market_money["Выручка"].mean():.2f}')
plt.legend()

plt.subplot(1, 2, 2)
sns.boxplot(data=market_money, x='Выручка')

# Анализ нулевых значений
print("\nАнализ нулевых и отрицательных значений:")
zero_revenue = market_money[market_money['Выручка'] == 0]
negative_revenue = market_money[market_money['Выручка'] < 0]
print(f"Записей с нулевой выручкой: {len(zero_revenue)} ({len(zero_revenue)/len(market_money)*100:.1f}%)")
print(f"Записей с отрицательной выручкой: {len(negative_revenue)}")
print(f"Клиентов с нулевой выручкой: {zero_revenue['id'].nunique()}")

### Анализ таблицы market_time

In [ ]:
# Статистика по времени на сайте
print("\nСтатистика по времени на сайте:")
print(f"Среднее время на сайте: {market_time['минут'].mean():.2f} минут")
print(f"Медианное время: {market_time['минут'].median():.2f} минут")
print(f"Максимальное время: {market_time['минут'].max():.2f} минут")
print(f"Минимальное время: {market_time['минут'].min():.2f} минут")
print(f"Стандартное отклонение: {market_time['минут'].std():.2f} минут")

# Гистограмма распределения времени
plt.figure(figsize=(30, 6))
plt.subplot(1, 2, 1)
sns.histplot(data=market_time, x='минут', kde=True, bins=50, color='green')
plt.title('Распределение времени на сайте')
plt.xlabel('Минуты на сайте')
plt.ylabel('Количество записей')
plt.axvline(market_time['минут'].mean(), color='red', linestyle='--', 
            label=f'Среднее: {market_time["минут"].mean():.2f}')
plt.legend()
plt.show()

Целевая переменная: Дисбаланс 38/62 (снизилась/прежний уровень). Нужна метрика F1-score.

Категориальные признаки: Большинство клиентов со "стандартным" сервисом, согласны на рассылки, чаще покупают товары для детей.

Числовые признаки: Разные масштабы и распределения, есть выбросы. Требуется масштабирование.

Активные клиенты: 1297 клиентов покупали хотя бы раз за 2 месяца. Их будем использовать для модели.

Данные подготовлены для объединения и моделирования. Есть дисбаланс классов, требуется масштабирование признаков.

## Объединение таблиц

In [ ]:
market_money_wide = market_money.pivot_table(
    index='id',
    columns='Период',
    values='Выручка'
) #преобразование market_money в широкий формат

market_money_wide.columns = [f'выручка_{col}' for col in market_money_wide.columns] #переименование

In [ ]:
market_money_wide.head()

In [ ]:
market_time_wide = market_time.pivot_table(
    index='id',
    columns='Период',
    values='минут'
) #преобразование market_time в широкий формат

market_time_wide.columns = [f'минут_{col}' for col in market_time_wide.columns]

In [ ]:
market_time_wide.head()

In [ ]:
merged_data = pd.merge(
    market_money_wide.loc[active_clients].reset_index(),  # Только активные клиенты!
    market_time_wide.reset_index(),
    on='id',
    how='inner'
)#объединение таблиц market_money_wide и market_time_wide

In [ ]:
merged_data = pd.merge(
    merged_data,
    market_file,
    on='id',
    how='inner'
) #объединение таблиц merged_data и market_file

In [ ]:
merged_data.head()

In [ ]:
merged_data.info()

In [ ]:
merged_data = merged_data.dropna().reset_index(drop=True)

In [ ]:
merged_data.isna().sum().sum()

## Корреляционный анализ

In [ ]:
# Создаем копию данных для корреляционного анализа
correlation_data = merged_data.copy()

# Преобразуем целевую переменную в числовой формат
if correlation_data['Покупательская_активность'].dtype == 'object':
    correlation_data['target_encoded'] = correlation_data['Покупательская_активность'].map(
        {'Снизилась': 1, 'Прежний уровень': 0}
    )
else:
    correlation_data['target_encoded'] = correlation_data['Покупательская_активность']

# Определяем интервальные признаки
interval_cols = correlation_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Убираем id из интервальных признаков, так как это идентификатор
if 'id' in interval_cols:
    interval_cols.remove('id')

# Убираем target_encoded из интервальных признаков, так как это идентификатор
if 'target_encoded' in interval_cols:
    interval_cols.remove('target_encoded')


# Вычисляем матрицу φK
phik_matrix = correlation_data.phik_matrix(interval_cols=interval_cols)

# Визуализация тепловой карты коэффициентов корреляции
print(interval_cols)

In [ ]:
plot_correlation_matrix(
    phik_matrix.values,
    x_labels=phik_matrix.columns,
    y_labels=phik_matrix.index,
    title=r"Матрица корреляции",
    fontsize_factor=1.5,
    figsize=(15, 12)
)


In [ ]:
interval_cols

In [ ]:
high_corr_pairs = []
for i in range(len(phik_matrix.columns)):
    for j in range(i+1, len(phik_matrix.columns)):
        corr_value = abs(phik_matrix.iloc[i, j])
        if corr_value > 0.7:  # Порог для сильной корреляции
            high_corr_pairs.append((
                phik_matrix.columns[i], 
                phik_matrix.columns[j], 
                phik_matrix.iloc[i, j]
            ))

print(f"\nНайдено пар признаков с корреляцией > |0.7|: {len(high_corr_pairs)}")

Максимальная корреляция составляет 0.75 между покупательской активностью и страниц за визит.

Вывод: Все признаки могут быть использованы в модели

## Использование пайплайнов

In [ ]:
data_for_model = merged_data.copy() #создадим копию данных

In [ ]:
data_for_model['target'] = data_for_model['Покупательская_активность'].map(
    {'Снизилась': 1, 'Прежний уровень': 0}
) #кодировка целевой переменной

X = data_for_model.drop(['id', 'Покупательская_активность', 'target'], axis=1)
y = data_for_model['target']

#Произведем деление на выборки (60/20/20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
) 

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

In [ ]:
# Определим категориальные и числовые признаки
cat_features = X.select_dtypes(include=['category', 'object']).columns.tolist()
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Разделяем категориальные признаки на бинарные (2 категории) и номинальные (>2 категорий)
binary_features = [col for col in cat_features if X[col].nunique() == 2]
nominal_features = [col for col in cat_features if X[col].nunique() > 2]

print(f"Бинарные категориальные признаки (2 категории): {binary_features}")
print(f"Номинальные категориальные признаки (>2 категорий): {nominal_features}")

In [ ]:
preprocessor_1 = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), cat_features)
])

transformers_2 = [('num', RobustScaler(), num_features)]

if binary_features:
    transformers_2.append(('cat_binary', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), binary_features))

if nominal_features:
    transformers_2.append(('cat_nominal', OneHotEncoder(handle_unknown='ignore', sparse=False), nominal_features))

preprocessor_2 = ColumnTransformer(transformers_2)

preprocessors = {
    'ohe_standard': preprocessor_1,
    'mixed_robust': preprocessor_2
}

In [ ]:
# Создаем пайплайны для каждой модели с обоими препроцессорами

#1. KNeighborsClassifier
knn_pipeline_1 = Pipeline([
    ('preprocessor', preprocessor_1),
    ('classifier', KNeighborsClassifier(n_neighbors=7))
])

knn_pipeline_2 = Pipeline([
    ('preprocessor', preprocessor_2),
    ('classifier', KNeighborsClassifier(n_neighbors=7))
])

In [ ]:
#2. DecisionTreeClassifier
tree_pipeline_1 = Pipeline([
    ('preprocessor', preprocessor_1),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

tree_pipeline_2 = Pipeline([
    ('preprocessor', preprocessor_2),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

In [ ]:
#3. LogisticRegression
logreg_pipeline_1 = Pipeline([
    ('preprocessor', preprocessor_1),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

logreg_pipeline_2 = Pipeline([
    ('preprocessor', preprocessor_2),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

In [ ]:
#4. SVC (Support Vector Classifier)
svc_pipeline_1 = Pipeline([
    ('preprocessor', preprocessor_1),
    ('classifier', SVC(random_state=42, probability=True))
])

svc_pipeline_2 = Pipeline([
    ('preprocessor', preprocessor_2),
    ('classifier', SVC(random_state=42, probability=True))
])

In [ ]:
pipelines = {
    'KNN_1': knn_pipeline_1,
    'KNN_2': knn_pipeline_2,
    'Tree_1': tree_pipeline_1,
    'Tree_2': tree_pipeline_2,
    'LogReg_1': logreg_pipeline_1,
    'LogReg_2': logreg_pipeline_2,
    'SVC_1': svc_pipeline_1,
    'SVC_2': svc_pipeline_2
}

Определение параметров для GridSearchCV

In [ ]:
# Параметры для GridSearch для каждой модели
knn_params = {
    'classifier__n_neighbors': [3, 5, 7, 9],
    'classifier__weights': ['uniform', 'distance']
}

tree_params = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10]
}

logreg_params = {
    'classifier__C': [0.1, 1, 10],
    'classifier__class_weight': [None, 'balanced']
}

svc_params = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__class_weight': [None, 'balanced']
}

# Создаем словарь параметров для каждого типа пайплайна
param_grids = {
    'KNN': knn_params,
    'Tree': tree_params,
    'LogReg': logreg_params,
    'SVC': svc_params
}

В данной задаче мы прогнозируем бинарную целевую переменную: снижение покупательской активности (класс 1) или сохранение прежнего уровня (класс 0).

Основная цель — выявить клиентов с риском снижения активности. Класс "снизилась" (положительный класс) критически важен для бизнеса. F1-binary фокусируется именно на этом классе.

Неравнозначная важность ошибок:

False Negative (пропуск клиента со снижением) → упущенная выгода

False Positive (ложное предсказание снижения) → неэффективные затраты

Оба типа ошибок важны, но для бизнеса критичнее FN

F1-binary обеспечивает баланс для целевого класса:

Precision: минимизация ложных срабатываний (экономия маркетингового бюджета)

Recall: максимизация обнаружения реальных случаев снижения активности

In [ ]:
# Для хранения результатов
results = []
# Обучение KNN моделей

for pipeline_name in ['KNN_1', 'KNN_2']:
    print(f"   - {pipeline_name}")
    grid_search = GridSearchCV(
        pipelines[pipeline_name],
        knn_params,
        cv=5,
        scoring='f1',
        n_jobs=-1,
    )
    grid_search.fit(X_train, y_train)
    
    # Оценка на валидации
    y_val_pred = grid_search.predict(X_val)
    f1_val = f1_score(y_val, y_val_pred, average='binary')
    
    results.append({
        'model': 'KNeighborsClassifier',
        'pipeline': pipeline_name,
        'best_params': grid_search.best_params_,
        'f1_val': f1_val,
        'best_estimator': grid_search.best_estimator_
    })
    print(f" F1-score на валидации: {f1_val:.4f}")

In [ ]:
# Обучение DecisionTree моделей
for pipeline_name in ['Tree_1', 'Tree_2']:
    print(f"   - {pipeline_name}")
    grid_search = GridSearchCV(
        pipelines[pipeline_name],
        tree_params,
        cv=5,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_train, y_train)
    
    # Оценка на валидации
    y_val_pred = grid_search.predict(X_val)
    f1_val = f1_score(y_val, y_val_pred, average='binary')
    
    results.append({
        'model': 'DecisionTreeClassifier',
        'pipeline': pipeline_name,
        'best_params': grid_search.best_params_,
        'f1_val': f1_val,
        'best_estimator': grid_search.best_estimator_
    })
    print(f"F1-score на валидации: {f1_val:.4f}")

In [ ]:
# Обучение LogisticRegression моделей
for pipeline_name in ['LogReg_1', 'LogReg_2']:
    print(f"   - {pipeline_name}")
    grid_search = GridSearchCV(
        pipelines[pipeline_name],
        logreg_params,
        cv=5,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_train, y_train)
    
    # Оценка на валидации
    y_val_pred = grid_search.predict(X_val)
    f1_val = f1_score(y_val, y_val_pred, average='binary')
    
    results.append({
        'model': 'LogisticRegression',
        'pipeline': pipeline_name,
        'best_params': grid_search.best_params_,
        'f1_val': f1_val,
        'best_estimator': grid_search.best_estimator_
    })
    print(f"F1-score на валидации: {f1_val:.4f}")

In [ ]:
#Обучение SVC моделей
for pipeline_name in ['SVC_1', 'SVC_2']:
    print(f"   - {pipeline_name}")
    grid_search = GridSearchCV(
        pipelines[pipeline_name],
        svc_params,
        cv=5,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_train, y_train)
    
    #Оценка на валидации
    y_val_pred = grid_search.predict(X_val)
    f1_val = f1_score(y_val, y_val_pred, average='binary')
    
    results.append({
        'model': 'SVC',
        'pipeline': pipeline_name,
        'best_params': grid_search.best_params_,
        'f1_val': f1_val,
        'best_estimator': grid_search.best_estimator_
    })
    print(f"F1-score на валидации: {f1_val:.4f}")

In [ ]:
#Создаем датафрем с результатами
results_df = pd.DataFrame(results)
print(results_df[['model', 'pipeline', 'f1_val']].sort_values('f1_val', ascending=False))

In [ ]:
#Находим лучший результат
best_result = results_df.loc[results_df['f1_val'].idxmax()]

print(f"Лучшая модель: {best_result['model']}")
print(f"Пайплайн: {best_result['pipeline']}")
print(f"Лучшие параметры: {best_result['best_params']}")
print(f"F1-score на валидации: {best_result['f1_val']:.4f}")

In [ ]:
#Сохраняем лучший пайплайн
best_pipeline = best_result['best_estimator']

y_test_pred = best_pipeline.predict(X_test)
y_test_proba = best_pipeline.predict_proba(X_test)[:, 1]

#Расчет метрик
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

test_f1 = f1_score(y_test, y_test_pred, average='binary')
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"F1-score на тесте: {test_f1:.4f}")
print(f"ROC-AUC на тесте: {test_auc:.4f}")

Созданы пайплайны для каждой из 4 моделей с 2 разными препроцессорами. Итого - 8 пайплайнов (4 модели × 2 препроцессора).

Подобраны гиперпараметры для каждой модели:

1. KNN: n_neighbors и weights

2. DecisionTree: max_depth и min_samples_split

3. LogisticRegression: C и class_weight

4. SVC: C, kernel и class_weight

Лучшая модель: KNeighborsClassifier и пайплайн KNN_1

F1-score на валидации: 0.833

F1-score на тесте: 0.8602

ROC-AUC на тесте: 0.9112

Мы получили достаточно высокие показатели для задачи классификации с дисбалансом классов. Модель эффективно находит баланс между: Precision и Recall. Модель успешно выявляет 90 % клиентов, требующих внимания, с хорошим балансом ошибок. Модель готова к применению в бизнес-процессах компании.

## Анализ важности признаков

In [ ]:
# Получаем препроцессор и модель из лучшего пайплайна
preprocessor = best_pipeline.named_steps['preprocessor']
model = best_pipeline.named_steps['classifier']

# Преобразуем данные для SHAP
X_val_transformed = preprocessor.transform(X_val)

# Используем очень маленькую выборку для SHAP (иначе будет очень долго)
sample_size = min(30, len(X_val_transformed))  # Уменьшили до 30 для скорости
X_sample = shap.utils.sample(X_val_transformed, sample_size)

In [ ]:
# Создаем функцию-предсказатель для SHAP
def predict_proba_wrapper(X):
    return model.predict_proba(X)

# Для KNN используем KernelExplainer
explainer = shap.KernelExplainer(predict_proba_wrapper, X_sample)
shap_values = explainer.shap_values(X_sample)

# Получаем имена признаков после преобразования (для старого sklearn)
feature_names = []

# Получаем компоненты препроцессора
transformers = preprocessor.named_transformers_

# 1. Обрабатываем числовые признаки
if 'num' in transformers:
    feature_names.extend(num_features)

# 2. Обрабатываем категориальные признаки
if 'cat' in transformers:
    cat_encoder = transformers['cat']
    

    if hasattr(cat_encoder, 'categories_'):
        for i, feature_name in enumerate(cat_features):
            categories = cat_encoder.categories_[i]
            for category in categories:
                # Преобразуем категорию в строку для безопасности
                category_str = str(category)
                category_str = category_str.replace(' ', '_').replace('/', '_')
                feature_names.append(f"{feature_name}_{category_str}")
    else:
        for feature_name in cat_features:
            # Оцениваем количество категорий
            unique_vals = X_val[feature_name].nunique()
            for i in range(unique_vals):
                feature_names.append(f"{feature_name}_cat{i}")

print(f"\nКоличество признаков: {len(feature_names)}")

# Визуализация SHAP
# Определяем, какие значения SHAP использовать
if isinstance(shap_values, list) and len(shap_values) == 2:
    # Для бинарной классификации используем значения для положительного класса
    shap_values_plot = shap_values[:,:,1]
else:
    shap_values_plot = shap_values[:,:,1]


In [ ]:
# Распределение влияния
fig2, ax2 = plt.subplots(figsize=(20, 14))
shap.summary_plot(shap_values_plot, X_sample, feature_names=feature_names, 
                  max_display=20, show=False, plot_size=(20, 15))
plt.title("Распределение влияния признаков на предсказание модели", 
          fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Средняя важность признаков
fig1, ax1 = plt.subplots(figsize=(18, 12))
shap.summary_plot(shap_values_plot, X_sample, feature_names=feature_names, 
                  plot_type="bar", max_display=20, show=False)
plt.title("Топ-20 наиболее важных признаков (среднее абсолютное значение SHAP)", 
          fontsize=18, fontweight='bold', pad=20)
plt.xlabel("Среднее абсолютное значение SHAP", fontsize=14)
plt.ylabel("Признаки", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Анализ важности в численном виде
mean_abs_shap = np.abs(shap_values_plot).mean(axis=0)

# Сортируем признаки по важности
sorted_indices = mean_abs_shap.argsort()[::-1]  # по убыванию

print("\nТОП-10 НАИБОЛЕЕ ВАЖНЫХ ПРИЗНАКОВ:")
print("-"*50)
for i, idx in enumerate(sorted_indices[:10]):
    print(f"{i+1:2}. {feature_names[idx]:30} : {mean_abs_shap[idx]:.5f}")

print("\nТОП-10 НАИМЕНЕЕ ВАЖНЫХ ПРИЗНАКОВ:")
print("-"*50)
for i, idx in enumerate(sorted_indices[-10:]):
    print(f"{i+1:2}. {feature_names[idx]:30} : {mean_abs_shap[idx]:.5f}")

Самые сильные драйверы прогноза модели: 
 1. Страниц_за_визит               : 0.09026
 2. минут_предыдущий_месяц         : 0.06726
 3. минут_текущий_месяц            : 0.05352
 4. Средний_просмотр_категорий_за_визит : 0.04614
 5. Акционные_покупки              : 0.03653

Анализ позволяет перейти от данных к конкретным действиям:

1. Сфокусироваться на «акционных» клиентах: Клиенты с долей покупок по акциям — группа наивысшего риска. Для них стоит разработать программу лояльности, постепенно вовлекая в неакционные покупки.

2. Мониторить активность просмотра: Падение среднего количества Страниц_за_визит и общего времени на сайте — оперативный сигнал для персонального контакта (e-mail, чат) с целью выяснить причины и предложить помощь.

3. Проанализировать долгосрочный тренд, а не только последний месяц: Финансовая активность за позапрошлый месяц оказалась более прогнозной. Это говорит о том, что решения об уходе клиент принимает не спонтанно, и у компании есть окно для манёвра.

## Сегментация покупателей

In [ ]:
segmentation_data = merged_data.copy()
segmentation_data['Вероятность_снижения'] = best_pipeline.predict_proba(segmentation_data.drop(['id', 'Покупательская_активность'], axis=1, errors='ignore'))[:, 1]

In [ ]:
segmentation_data = segmentation_data.merge(money[['id', 'Прибыль']], on='id', how='left')

print(f"Клиентов для сегментации: {len(segmentation_data)}")
print(f"Пропуски в прибыли: {segmentation_data['Прибыль'].isna().sum()}")

In [ ]:
# Определяем пороги для сегментации
risk_threshold = segmentation_data['Вероятность_снижения'].median()  # медиана вероятности
value_threshold = segmentation_data['Прибыль'].median()  # медиана прибыли

print("Пороги для сегментации:")
print(f"  Риск (вероятность снижения): {risk_threshold:.3f}")
print(f"  Ценность (прибыль): {value_threshold:.3f}")

In [ ]:
# Создаем сегменты
def assign_segment(row):
    if row['Вероятность_снижения'] >= risk_threshold and row['Прибыль'] >= value_threshold:
        return 'Высокий риск, Высокая ценность'
    elif row['Вероятность_снижения'] >= risk_threshold and row['Прибыль'] < value_threshold:
        return 'Высокий риск, Низкая ценность'
    elif row['Вероятность_снижения'] < risk_threshold and row['Прибыль'] >= value_threshold:
        return 'Низкий риск, Высокая ценность'
    else:
        return 'Низкий риск, Низкая ценность'

segmentation_data['Сегмент'] = segmentation_data.apply(assign_segment, axis=1)

# Анализируем распределение по сегментам
segment_stats = segmentation_data['Сегмент'].value_counts()
print(f"\nРаспределение клиентов по сегментам:")
for segment, count in segment_stats.items():
    percentage = count / len(segmentation_data) * 100
    print(f"  {segment}: {count} клиентов ({percentage:.1f}%)")

In [ ]:
# Анализ средних значений ключевых показателей по сегментам
key_features = ['Акционные_покупки', 'Страниц_за_визит', 'минут_текущий_месяц', 
                'выручка_препредыдущий_месяц', 'Маркет_актив_6_мес', 'Длительность']

segment_analysis = segmentation_data.groupby('Сегмент')[key_features + ['Прибыль', 'Вероятность_снижения']].mean()

In [ ]:
# Визуализация сравнения сегментов по ключевым признакам
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, feature in enumerate(key_features):
    segment_means = segmentation_data.groupby('Сегмент')[feature].mean()
    colors = ['red', 'orange', 'green', 'blue']
    bars = axes[i].bar(segment_means.index, segment_means.values, color=colors)
    axes[i].set_title(feature, fontsize=12)
    axes[i].set_xticklabels(segment_means.index, rotation=45, ha='right')
    axes[i].grid(axis='y', alpha=0.3)
    
    # Добавляем значения
    for bar in bars:
        height = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                    f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Выделяем сегмент "Высокий риск, Высокая ценность" для детального анализа
high_risk_high_value = segmentation_data[segmentation_data['Сегмент'] == 'Высокий риск, Высокая ценность'].copy()

print(f"ДЕТАЛЬНЫЙ АНАЛИЗ СЕГМЕНТА: ВЫСОКИЙ РИСК, ВЫСОКАЯ ЦЕННОСТЬ")
print(f"Клиентов в сегменте: {len(high_risk_high_value)}")
print(f"Доля от всех клиентов: {len(high_risk_high_value)/len(segmentation_data)*100:.1f}%")

print(f"\nКлючевые характеристики сегмента:")
print(f"Средняя вероятность снижения: {high_risk_high_value['Вероятность_снижения'].mean():.3f}")
print(f"Средняя прибыль: {high_risk_high_value['Прибыль'].mean():.3f}")
print(f"Средняя доля акционных покупок: {high_risk_high_value['Акционные_покупки'].mean():.3f}")
print(f"Среднее время на сайте: {high_risk_high_value['минут_текущий_месяц'].mean():.1f} мин")

# Сравнение с общими средними
print(f"\nСравнение с общей выборкой:")
for feature in ['Акционные_покупки', 'Страниц_за_визит', 'минут_текущий_месяц', 'Прибыль']:
    segment_mean = high_risk_high_value[feature].mean()
    overall_mean = segmentation_data[feature].mean()
    difference = segment_mean - overall_mean
    difference_pct = (difference / overall_mean) * 100 if overall_mean != 0 else 0
    print(f"  {feature}: {segment_mean:.3f} vs {overall_mean:.3f} ({difference_pct:+.1f}%)")

# Распределение по популярным категориям в этом сегменте
print(f"\nТоп-5 популярных категорий в сегменте:")
top_categories = high_risk_high_value['Популярная_категория'].value_counts().head()
for category, count in top_categories.items():
    percentage = count / len(high_risk_high_value) * 100
    print(f"  {category}: {count} клиентов ({percentage:.1f}%)")

Сегментация выполнена успешно: Клиенты разделены на 4 сегмента по матрице "Риск-Ценность" с использованием:

Риск: Вероятность снижения активности (прогноз модели KNN)

Ценность: Прибыль от клиента (данные financial department)

Ключевые сегменты:

Высокий риск, Высокая ценность (25,3% клиентов) - приоритет для удержания

Высокий риск, Низкая ценность (25,3% клиентов) - стандартные меры удержания

Низкий риск, Высокая ценность (24,7% клиентов) - развитие лояльности

Низкий риск, Низкая ценность (24,7% клиентов) - массовый маркетинг

Для сегмента "Высокий риск, Высокая ценность" предложены конкретные меры:

1. Персонализированные предложения для снижения акционной зависимости

2. Программа VIP-обслуживания

3. Увеличение вовлеченности через контент и сервис

Практическая ценность: сегментация позволяет:

1. Рационально распределить маркетинговый бюджет

2. Сфокусировать усилия на самых ценных клиентах

3. Разработать таргетированные стратегии для каждой группы

## Общий вывод

Была решена бизнес-задача по удержанию клиентов для интернет-магазина «В один клик». Целью было разработать решение для прогнозирования снижения покупательской активности и на его основе создать персонализированные стратегии для ключевых сегментов клиентов.

В качестве исходных данных  4 таблицы с данными о клиентах, их поведении на сайте, покупках, коммуникациях и прибыльности. Их объем составляет 1300 клиентов с историей за 3 месяца.

В рамках работы было выполнено преобразование типов данных (исправление Прибыль из object в float), формата данных из "длинного" в "широкий", фильтрация активных клиентов (совершавших покупки хотя бы раз за 3 месяца), кодирование категориальных признаков, масштабирование числовых признаков.

Для выбора лучшей модели:

Созданы пайплайны с двумя вариантами предобработки: OneHotEncoder + StandardScaler и OrdinalEncoder + RobustScaler.

Протестированы 4 модели с подбором гиперпараметров:

KNeighborsClassifier (n_neighbors, weights)

DecisionTreeClassifier (max_depth, min_samples_split)

LogisticRegression (C, class_weight)

SVC (C, kernel, class_weight)

Метрика качества: Использован F1-score (macro) как оптимальная метрика для задачи с дисбалансом классов (38%/62%).

В результате лучшая модель- KNeighborsClassifier с препроцессором OrdinalEncoder + RobustScaler показала наилучшие результаты:

F1-score на тестовой выборке: 0.8996

ROC-AUC на тесте: 0.91

Интерпретируемость: Модель эффективно выявляет клиентов группы риска с хорошим балансом точности и полноты.

Метод перестановочной важности выявил ключевые драйверы снижения активности: Акционные_покупки, Страниц_за_визит, минут_предыдущий_месяц. Клиенты с высокой долей акционных покупок и снижающейся активностью на сайте — группа максимального риска.

На основе прогнозов модели и данных о прибыльности клиенты разделены на 4 сегмента по матрице "Риск-Ценность". Приоритетный сегмент для удержания: «Высокий риск — Высокая ценность» (20-25% клиентов)

Конкретные предложения для этого сегмента:

1. Персонализированные коммерческие предложения: «мягкий» уход от акционной зависимости- персональная скидка 10-15% на регулярные (неакционные) товары из категорий, которые клиент уже покупал.

2. Увеличение вовлечённости: контент-маркетинг- отправка персонализированных подборок, обзоров и приглашений на онлайн-мероприятия, связанные с их интересами.

3. Мониторинг и оценка. Внедрить KPI- снижение доли акционных покупок в сегменте на 15% за квартал при сохранении выручки.


Модель требует регулярного переобучения на свежих данных (раз в 1-3 месяца). Выявлены корреляции, а не причинно-следственные связи. Рекомендуется провести качественное исследование (опросы, интервью) для понимания мотивов клиентов.